# run_all — Full Pipeline Orchestrator

**Jackson and Johnson HR Digital**

Runs every pipeline notebook in order, mirroring the production job DAG.
Use this for interactive end-to-end runs directly from the workspace — the DAG in
`databricks.yml` handles scheduled / CI execution.

### Execution order
```
00_load_bronze
    ├── 01_load_silver          ┐  (parallel)
    └── 01b_build_vector_search ┘
         └── 02_apply_data_quality_and_classification
                  └── 03_build_gold
                           ├── 03b_apply_business_semantics
                           │         └── 04_create_genie_space  ┐  (parallel)
                           └── 04b_create_uc_functions          ┘
                                    └── 05_train_ml_model  (skipped if train_model=false)
                                             └── 06_evaluate_register_agent
                                                      └── 07_deploy_app
                                                               └── 08_refresh_dashboard
```

In [ ]:
dbutils.widgets.text("catalog",              "bx4",                             "UC Catalog")
dbutils.widgets.text("schema",               "hrd_2030",                        "UC Schema")
dbutils.widgets.text("volume_name",          "raw_data",                        "UC Volume Name")
dbutils.widgets.text("warehouse_id",         "aa06644a4b0be047",                "SQL Warehouse ID")
dbutils.widgets.text("vs_endpoint_name",     "bx4_hrd_vs_endpoint",             "VS Endpoint Name")
dbutils.widgets.text("vs_index_name",        "hr_resumes_vs_index",             "VS Index Name")
dbutils.widgets.text("genie_space_id",       "01f13a0f6a081fabbea933cfb0db1d01", "Genie Space ID (leave blank to auto-create)")
dbutils.widgets.text("mlflow_experiment",    "ml-hr-predictive-hiring-evals",   "MLflow Experiment Name (ML model, nb 05)")
dbutils.widgets.text("agent_mlflow_experiment", "agent-predictive-hiring-evals", "MLflow Experiment Name (Agent, nb 06)")
dbutils.widgets.text("model_name",           "ml_hr_predictive_hiring",         "ML Model Name")
dbutils.widgets.text("model_endpoint_name",  "hr-predictive-hiring-endpoint",   "Model Endpoint Name")
dbutils.widgets.text("agent_name",           "agent-hire_right",                "Agent Model Name")
dbutils.widgets.text("agent_endpoint_name",  "hire-right-agent-endpoint",       "Agent Endpoint Name")
dbutils.widgets.text("llm_endpoint",         "databricks-gpt-5-4",              "LLM Endpoint")
dbutils.widgets.text("app_name",             "hire-right-agent-app",            "Databricks App Name")
dbutils.widgets.text("train_model",          "false",                           "Train ML model? (true/false)")
dbutils.widgets.text("deploy_agent",         "true",                            "Deploy agent? (true/false)")

In [ ]:
%pip install python-dotenv -q

In [ ]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

# Resolve notebook dir and load .env
_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_nb_path = _ctx.notebookPath().get()
NB_DIR       = _nb_path.rsplit("/", 1)[0]
project_root = "/Workspace" + _nb_path.rsplit("/notebooks", 1)[0]
load_dotenv(f"{project_root}/.env")

# Widget takes priority (job passes it); fall back to .env for interactive runs
catalog                = dbutils.widgets.get("catalog")                or os.getenv("TARGET_CATALOG", "bx4")
schema                 = dbutils.widgets.get("schema")                 or os.getenv("TARGET_SCHEMA", "hrd_2030")
volume_name            = dbutils.widgets.get("volume_name")
warehouse_id           = dbutils.widgets.get("warehouse_id")           or os.getenv("DATABRICKS_WAREHOUSE_ID", "")
vs_endpoint_name       = dbutils.widgets.get("vs_endpoint_name")       or os.getenv("VS_ENDPOINT_NAME", "bx4_hrd_vs_endpoint")
vs_index_name          = dbutils.widgets.get("vs_index_name")          or os.getenv("VS_INDEX", "hr_resumes_vs_index")
genie_space_id         = dbutils.widgets.get("genie_space_id")         or os.getenv("GENIE_SPACE_ID", "")
mlflow_experiment      = dbutils.widgets.get("mlflow_experiment")      or os.getenv("MLFLOW_EXPERIMENT_ML", "ml-hr-predictive-hiring-evals")
agent_mlflow_experiment = dbutils.widgets.get("agent_mlflow_experiment") or os.getenv("MLFLOW_EXPERIMENT_AGENT", "agent-predictive-hiring-evals")
model_name             = dbutils.widgets.get("model_name")
model_endpoint_name    = dbutils.widgets.get("model_endpoint_name")    or os.getenv("MODEL_ENDPOINT_NAME", "hr-predictive-hiring-endpoint")
agent_name             = dbutils.widgets.get("agent_name")             or os.getenv("AGENT_NAME", "hire_right")
agent_endpoint_name    = dbutils.widgets.get("agent_endpoint_name")    or os.getenv("DATABRICKS_AGENT_ENDPOINT", "hire-right-agent-endpoint")
llm_endpoint           = dbutils.widgets.get("llm_endpoint")           or os.getenv("LLM_ENDPOINT_NAME", "databricks-gpt-5-4")
app_name               = dbutils.widgets.get("app_name")               or os.getenv("APP_NAME", "hire-right-agent-app")
train_model            = dbutils.widgets.get("train_model").strip().lower() == "true"
deploy_agent           = dbutils.widgets.get("deploy_agent").strip().lower() == "true"

def nb(name):
    """Return the workspace path for a sibling notebook."""
    return f"{NB_DIR}/{name}"

print(f"Catalog              : {catalog}")
print(f"Schema               : {schema}")
print(f"Warehouse ID         : {warehouse_id or '(from .env or auto-detect)'}")
print(f"VS Endpoint          : {vs_endpoint_name}")
print(f"Train ML model       : {train_model}")
print(f"Deploy agent         : {deploy_agent}")
print(f"MLflow Exp (ML)      : {mlflow_experiment}")
print(f"MLflow Exp (Agent)   : {agent_mlflow_experiment}")
print(f"Notebook dir         : {NB_DIR}")

In [ ]:
# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def run(notebook_name, params=None, timeout=3600):
    """Run a sibling notebook and print a status line."""
    params = params or {}
    start  = time.time()
    print(f"  ▶  {notebook_name} ...", end="", flush=True)
    result = dbutils.notebook.run(nb(notebook_name), timeout, params)
    elapsed = int(time.time() - start)
    print(f" ✅ ({elapsed}s)")
    return result

def run_parallel(tasks):
    """Run a list of (notebook_name, params) tuples in parallel threads.
    Returns a dict of {notebook_name: result_string} for all successful tasks."""
    errors  = {}
    results = {}
    with ThreadPoolExecutor(max_workers=len(tasks)) as pool:
        futures = {
            pool.submit(run, name, params): name
            for name, params in tasks
        }
        for future in as_completed(futures):
            name = futures[future]
            try:
                results[name] = future.result()
            except Exception as e:
                errors[name] = str(e)
                print(f"  ❌ {name} FAILED: {e}")
    if errors:
        raise RuntimeError(f"Parallel tasks failed: {list(errors.keys())}")
    return results

# Common parameters passed to every notebook
base = {"catalog": catalog, "schema": schema}

In [ ]:
print("=" * 60)
print("Step 00 — Load Bronze")
print("=" * 60)
run("00_load_bronze", {**base, "volume_name": volume_name})

In [ ]:
print("=" * 60)
print("Step 01 — Load Silver + Build Vector Search (parallel)")
print("=" * 60)
run_parallel([
    ("01_load_silver", base),
    ("01b_build_vector_search", {
        **base,
        "volume_name":      volume_name,
        "vs_endpoint_name": vs_endpoint_name,
        "vs_index_name":    vs_index_name,
    }),
])

In [ ]:
print("=" * 60)
print("Step 02 — Apply Data Quality & Classification")
print("=" * 60)
run("02_apply_data_quality_and_classification", base)

In [ ]:
print("=" * 60)
print("Step 03 — Build Gold + Apply Business Semantics")
print("=" * 60)
run("03_build_gold", base)
run("03b_apply_business_semantics", base)

In [ ]:
print("=" * 60)
print("Step 04 — Create Genie Space + UC Functions (parallel)")
print("=" * 60)

results = run_parallel([
    ("04_create_genie_space",   {**base, "warehouse_id": warehouse_id}),
    ("04b_create_uc_functions", base),
])

# Capture the space ID returned by notebook 04 if not already set via widget/.env
if not genie_space_id:
    captured = (results.get("04_create_genie_space") or "").strip()
    if captured:
        genie_space_id = captured
        print(f"  Captured Genie Space ID: {genie_space_id}")

In [ ]:
print("=" * 60)
print(f"Step 05 — Train ML Model (train_model={train_model})")
print("=" * 60)
if train_model:
    run("05_train_ml_model", {
        **base,
        "mlflow_experiment": mlflow_experiment,
        "model_name":        model_name,
        "endpoint_name":     model_endpoint_name,
    })
else:
    print("  ⏭  Skipped (train_model=false)")

In [ ]:
print("=" * 60)
print(f"Step 06 — Evaluate & Register Agent (deploy_agent={deploy_agent})")
print("=" * 60)
if deploy_agent:
    result_06 = run("06_evaluate_register_agent", {
        **base,
        "genie_space_id":      genie_space_id,
        "vs_index_name":       vs_index_name,
        "vs_endpoint_name":    vs_endpoint_name,
        "model_endpoint_name": model_endpoint_name,
        "agent_name":          agent_name,
        "agent_endpoint_name": agent_endpoint_name,
        "llm_endpoint":        llm_endpoint,
        "mlflow_experiment":   agent_mlflow_experiment,
    })
    if result_06 and result_06.strip():
        agent_endpoint_name = result_06.strip()
        print(f"  Agent endpoint: {agent_endpoint_name}")
else:
    print("  ⏭  Skipped (deploy_agent=false)")

In [ ]:
print("=" * 60)
print(f"Step 07 — Deploy App (deploy_agent={deploy_agent})")
print("=" * 60)
if deploy_agent:
    run("07_deploy_app", {
        **base,
        "agent_endpoint_name": agent_endpoint_name,
        "warehouse_id":        warehouse_id,
        "app_name":            app_name,
        "genie_space_id":      genie_space_id,
    })
else:
    print("  ⏭  Skipped (deploy_agent=false)")

In [ ]:
print("=" * 60)
print("Step 08 — Refresh Dashboard")
print("=" * 60)
run("08_refresh_dashboard", {
    **base,
    "warehouse_id":   warehouse_id,
    "dashboard_name": "Jackson and Jackson HR Digital — Hiring Analytics",
})

if deploy_agent:
    print("=" * 60)
    print("Step 09 — Grant App Permissions")
    print("=" * 60)
    run("09_grant_app_permissions", {
        **base,
        "app_name":            app_name,
        "genie_space_id":      genie_space_id,
        "warehouse_id":        warehouse_id,
        "agent_endpoint_name": agent_endpoint_name,
    })
else:
    print("  ⏭  Step 09 skipped (deploy_agent=false)")

In [ ]:
print()
print("=" * 60)
print("✅  Jackson and Jackson HR Digital Pipeline Complete")
print("=" * 60)